SVM (Gridsearch + GroupKfold)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import os
import re
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

#1.------前処理------

#表情データの読み込みとdfに格納
folder_path = Path('../../../output_filtered_questions_copy')

file_list = folder_path.glob('*.csv')

file_path_list = [str(p) for p in file_list]

df = pd.DataFrame(file_path_list, columns=['filepath'])

files_df = df
files_df['filename'] = df['filepath'].apply(lambda p: Path(p).name)
files_df['ID'] = files_df['filename'].str.extract(r'ID(\d+)')
files_df['ID'] = files_df['ID'].astype(int)

#患者の疾患有無
labels_df = pd.read_csv(r"C:\Users\robotics\proj\Research\code\voice\delirium.csv")
#患者と疾患有無を結合
file_with_labels_df = pd.merge(files_df, labels_df, on='ID', how='left')
file_with_labels_df.drop(columns=['Sex'], inplace=True)

#print(file_with_labels_df)

#学習・テストデータとなるファイルの分割
X_files = file_with_labels_df.drop(columns='Delirum', errors='ignore')
#print(X_files)
y_labels = file_with_labels_df.drop(columns=['filepath', 'filename', 'ID'], errors='ignore')
#rint(y_labels)

#層化５分割交差検証
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(skf.split(X_files, y_labels)):
    X_train_sort, X_test_sort = X_files.iloc[train_index], X_files.iloc[test_index]
    y_train_sort, y_test_sort = y_labels.iloc[train_index], y_labels.iloc[test_index]

# #学習・テストデータとなるファイルの分割
# X_files = file_with_labels_df.drop(columns='Delirum', errors='ignore')
# #print(X_files)
# y_labels = file_with_labels_df.drop(columns=['filepath', 'filename'], errors='ignore')
# #print(y_labels)
# #80%学習データ、20%テストデータに分割（患者単位）
# X_train, X_test, y_train, y_test = train_test_split(
#     X_files, y_labels, test_size=0.2, random_state=42, stratify=y_labels['Delirium']
# )

# #print(X_train)
# #print(y_train)

# #順番をソート
# X_train_sort = X_train.sort_values(by='ID', ascending=True)
# X_test_sort = X_test.sort_values(by='ID', ascending=True)
# y_train_sort = y_train.sort_values(by='ID', ascending=True).drop(columns='ID', errors='ignore')
# y_test_sort = y_test.sort_values(by='ID', ascending=True).drop(columns='ID', errors='ignore')
#print(X_test_sort)

    #2.------学習データを読み込み、質問ごとの最大値を計算------

    train_file_path = [f for f in X_train_sort['filepath']]
    #print(train_file_path)
    train_df = pd.DataFrame()
    for f, a in zip(train_file_path, y_train_sort['Delirium']):
        df_train_file = pd.read_csv(f)
        #print(df_train_file)
        #ファイル名とラベルの付与
        current_filename = os.path.basename(f)
        #ファイル名の数字部分を抽出
        number = re.findall(r'\d+', current_filename)
        #print(number)
        df_train_file['Delirium'] = a
        df_train_file['filenumber'] = number[0]
        #print(df_train_file)  
        
        #必要のない列の削除
        #label列にある空白の削除
        df_train_file['label'] = df_train_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_train_sort_df = df_train_file[df_train_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_train_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_train_sort_df[confidence].index
        df_train_new_df = df_train_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_train_new_df.columns if col.startswith("AU")] + ["Delirium", "filenumber"]
        df_train_final_df = df_train_new_df[columns_to_extract]

        #print(df_train_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_train_final_df = df_train_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_train_final_df)

        #数値型に変更
        #df_train_final_df['filename'] = pd.to_numeric(df_train_final_df['filename'], errors='coerce')
        #print(df_train_final_df)

        #質問ごとの最大値を計算
        df_train_max_df = df_train_final_df.groupby('label').max()
        df_train_max_df = df_train_max_df.reset_index(drop=True)
        #print(df_train_max_df)

        train_df = pd.concat([train_df, df_train_max_df], ignore_index=True)

    #print(train_df)

    #3.------テストデータを読み込み、質問ごとの最大値を計算------

    test_file_path = [f for f in X_test_sort['filepath']]
    #print(test_file_path)
    test_df = pd.DataFrame()
    test_file_lengths = []
    for f, a in zip(test_file_path, y_test_sort['Delirium']):
        df_test_file = pd.read_csv(f)
        #print(df_test_file)
        #ファイル名とラベルの付与
        current_filename = os.path.basename(f)
        #print(current_filename)
        df_test_file['Delirium'] = a
        df_test_file['filename'] = current_filename
        #print(df_test_file)  
        
        #必要のない列の削除
        #label列にある空白の削除
        df_test_file['label'] = df_test_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_test_sort_df = df_test_file[df_test_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_test_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_test_sort_df[confidence].index
        df_test_new_df = df_test_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_test_new_df.columns if col.startswith("AU")] + ["Delirium", "filename"]
        df_test_final_df = df_test_new_df[columns_to_extract]

        #print(df_test_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_test_final_df = df_test_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_test_final_df)

        #数値型に変更
        #df_test_final_df['filename'] = pd.to_numeric(df_test_final_df['filename'], errors='coerce')
        #print(df_test_final_df)

        #質問ごとの最大値を計算
        df_test_max_df = df_test_final_df.groupby('label').max()
        df_test_max_df = df_test_max_df.reset_index(drop=True)
        #print(df_test_max_df)
        
        test_file_lengths.append(len(df_test_max_df))

        test_df = pd.concat([test_df, df_test_max_df], ignore_index=True)

    #print(test_df)
    #print(test_file_lengths)

    #4.------グリッドサーチのための学習・検証データの用意とテストデータの用意------

    y_train_grid = train_df['Delirium']
    y_test_grid = test_df['Delirium']

    columns_to_delete = ['Delirium']
    columns_to_delete2 = ['Delirium', 'filename']

    X_train_grid = train_df.drop(columns=columns_to_delete, errors='ignore')
    X_test_grid = test_df.drop(columns=columns_to_delete2, errors='ignore')




    #print(y_train_grid)

    #5.------学習の準備(学習器と学習方法の設定)------

    #患者ごとに分割するための変数を用意し、学習データからfilenumberを削除
    patient_groups = X_train_grid['filenumber'].values
    X_train_model = X_train_grid.drop(columns=['filenumber']).copy()
    #print(X_train_grid)
    #print(X_train_model)
    #モデル・グリッドサーチの定義
    model = SVC()
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    param_grid = {'kernel': ['linear', 'rbf'],
                'C': [0.01, 0.1, 1, 10, 100],
                'gamma': ['scale', 'auto', 0.1, 1] # rbfカーネルの形状を決める重要なパラメータ
    }
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='roc_auc', cv=cv, n_jobs=-1)

    #6.------グリッドサーチ------
    grid_search.fit(X_train_model, y_train_grid, groups=patient_groups)

    print(f"最適なハイパーパラメータ：{grid_search.best_params_}")
    print(f"CVでのmax精度(Accuracy):{grid_search.best_score_:.4f}")

    #行と列の表示制限を解除（または十分に大きな値に設定）
    pd.set_option('display.max_rows', None)     # すべての行を表示
    pd.set_option('display.max_columns', None)  # すべての列を表示
    pd.set_option('display.width', 1000)        # 表示幅を広くする

    #結果をDataFrameに変換し表示
    results_df = pd.DataFrame(grid_search.cv_results_)

    print("--- グリッドサーチの全試行結果 ---")
    print(results_df)

    #表示設定を元に戻す (推奨)
    #グローバルな設定を元に戻し、他の処理に影響を与えないようにします
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')


    #7.------テストデータでの性能評価------
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_grid)
    final_accuracy = accuracy_score(y_test_grid, y_pred)

    print(f"テストデータでの正解率: {final_accuracy:.4}\n")
    #print(y_pred)

    file_predictions = []
    start_idx = 0
    for length in test_file_lengths:
        end_idx = start_idx + length
        predictions_for_this_file = y_pred[start_idx:end_idx]
        majority_vote = mode(predictions_for_this_file)[0]
        file_predictions.append([majority_vote, start_idx, end_idx])
        start_idx = end_idx

    first_column_list = [item[0] for item in file_predictions]

    #print(first_column_list)

    y_test_files = X_test_sort['Delirium'].values

    #print(type(y_test_files))

    y_test_files_copy = X_test_sort

    y_test_files_copy = y_test_files_copy.assign(Predictions=first_column_list)

    print("各テストファイルごとの分類結果\n")
    print(y_test_files_copy.drop(columns=['filepath'], errors='ignore'))

    correct_files = np.sum(np.array(first_column_list) == y_test_files)
    total_files = len(X_test_sort)
    file_accuracy = correct_files/ total_files

    scores = []
    scores.append(file_accuracy)
    print("\n")
    print(f"正解率 (ファイル単位): {file_accuracy:.4f} ({correct_files}/{total_files} ファイル正解)\n")


    #8.------混同行列の作成------
    cm = confusion_matrix(y_test_files, first_column_list)
    cm_df = pd.DataFrame(cm, 
                        index=['正解: 0', '正解: 1'], 
                        columns=['予測: 0', '予測: 1'])

    print("混同行列:")
    print(cm_df)
    print("\n") # 見やすいように改行





最適なハイパーパラメータ：{'C': 10, 'gamma': 1, 'kernel': 'rbf'}
CVでのmax精度(Accuracy):0.5922
--- グリッドサーチの全試行結果 ---
    mean_fit_time  std_fit_time  mean_score_time  std_score_time  param_C param_gamma param_kernel                                             params  split0_test_score  split1_test_score  split2_test_score  split3_test_score  split4_test_score  mean_test_score  std_test_score  rank_test_score
0        0.008431      0.001217         0.005610        0.000789     0.01       scale       linear  {'C': 0.01, 'gamma': 'scale', 'kernel': 'linear'}           0.193738           0.501377           0.484736           0.526203           0.472710         0.435753        0.122331               31
1        0.010371      0.001310         0.007688        0.000845     0.01       scale          rbf     {'C': 0.01, 'gamma': 'scale', 'kernel': 'rbf'}           0.201076           0.457759           0.644311           0.474866           0.583333         0.472269        0.152116               24
2        0.008